In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Membaca dataset langsung dari file yang diupload di Google Colab
# Pastikan nama file sesuai dengan file spreadsheet survei yang kamu miliki
try:
    df = pd.read_excel("/content/Survei Beban Tugas & Tingkat Stres Mahasiswa  (Jawaban).xlsx")
    print("Dataset berhasil dimuat!")
except Exception as e:
    print(f"File tidak ditemukan atau terjadi kesalahan: {e}")
    print("Pastikan file sudah di-upload ke Google Colab dan nama file benar.")

# 2. Data Cleaning & Preprocessing
print(f"Initial DataFrame shape: {df.shape}")
print("Missing values before cleaning:")
print(df.isnull().sum())

# Drop 'Tes' column as it has all missing values and is not used for analysis
if 'Tes' in df.columns:
    df.drop('Tes', axis=1, inplace=True)
    print("Dropped 'Tes' column.")

# Rename columns for easier access
df.rename(columns={
    'Saat ini, ada berapa total tugas kuliah kamu yang masih aktif atau belum selesai?  ': 'Jumlah tugas',
    'Dari tugas-tugas tersebut, berapa jumlah tugas yang deadline-nya mepet (kurang dari 3 hari)?': 'Jumlah tugas melewati deadline',
    'Berapa rata-rata jam tidur kamu per hari dalam seminggu terakhir?  ': 'Rata-rata jam tidur',
    'Secara keseluruhan, apakah kamu merasa stres dengan beban akademikmu saat ini?  ': 'Status stress'
}, inplace=True)

# Now apply dropna and drop_duplicates after cleaning the problematic 'Tes' column and renaming.
df.dropna(inplace=True)  # Hapus missing values
df.drop_duplicates(inplace=True)  # Hapus data duplikat

print(f"DataFrame shape after cleaning and renaming: {df.shape}")

# Check if DataFrame is empty after cleaning
if df.empty:
    print("DataFrame is empty after cleaning. Please check your data for excessive missing values or duplicates.")
else:
    print("Unique values in 'Status stress' before mapping:")
    print(df['Status stress'].unique())

    # Ensure 'Status stress' column is string type and strip whitespace
    df['Status stress'] = df['Status stress'].astype(str).str.strip()

    # Mapping label target (Teks ke Biner)
    # Corrected mapping based on unique values observed: ['Ya', 'Tidak', 'Stress']
    df['Status stress'] = df['Status stress'].map({'Stres': 1, 'Tidak Stres': 0, 'Ya': 1, 'Tidak': 0, 'Stress': 1})

    print("Unique values in 'Status stress' after mapping:")
    print(df['Status stress'].unique())

    # Drop rows where 'Status stress' became NaN after mapping (if any)
    if df['Status stress'].isnull().any():
        initial_rows = len(df)
        df.dropna(subset=['Status stress'], inplace=True)
        print(f"Dropped {initial_rows - len(df)} rows with NaN in 'Status stress' after mapping.")

    # 3. Menentukan Fitur (X) dan Target (y)
    X = df[['Jumlah tugas', 'Jumlah tugas melewati deadline', 'Rata-rata jam tidur']]
    y = df['Status stress']

    # Check if X and y are empty before splitting
    if X.empty or y.empty:
        print("Features (X) or target (y) are empty after column selection. Please check column names and data.")
    else:
        # 4. Membagi data menjadi 80% Training dan 20% Testing
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 5. Membangun dan Melatih Model Logistic Regression
        model = LogisticRegression()
        model.fit(X_train, y_train)

        # 6. Melakukan Prediksi Klasifikasi dan Probabilitas
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)

        # 7. Menampilkan Hasil Evaluasi Model
        print("\n=== EVALUASI MODEL REGRESI LOGISTIK ===")
        print(f"Akurasi Model: {accuracy_score(y_test, y_pred) * 100:.2f}%")

        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_pred))

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        print("\n=== CONTOH PROBABILITAS PREDIKSI (5 DATA PERTAMA UJI) ===")
        for i in range(min(5, len(X_test))):
            print(f"Data ke-{i+1} -> Probabilitas [Tidak Stres, Stres]: {y_prob[i]} -> Prediksi: {y_pred[i]}")

Dataset berhasil dimuat!
Initial DataFrame shape: (40, 8)
Missing values before cleaning:
Timestamp                                                                                        3
Nama Lengkap                                                                                     3
NIM                                                                                              3
Saat ini, ada berapa total tugas kuliah kamu yang masih aktif atau belum selesai?                3
Dari tugas-tugas tersebut, berapa jumlah tugas yang deadline-nya mepet (kurang dari 3 hari)?     3
Berapa rata-rata jam tidur kamu per hari dalam seminggu terakhir?                                3
Secara keseluruhan, apakah kamu merasa stres dengan beban akademikmu saat ini?                   3
Tes                                                                                             40
dtype: int64
Dropped 'Tes' column.
DataFrame shape after cleaning and renaming: (37, 7)
Unique values in 'Status stres